# Path Corpus Analysis — Language & MD Networks

This notebook builds a **probabilistic corpus of shortest paths** across all AAL regions,
using large-scale fMRI data from Diachek et al. (2020) and Lipkin et al. (2022). The
`NETWORK` toggle in Section 2 selects which of the two fMRI datasets the corpus is built from.

The **path corpus** captures, for every ordered pair of AAL regions:

- which shortest paths are actually taken by healthy participants (`path_counts`),
- how discoverable each path is under a random walk (`path_prob`, via search information),
- how often each intermediate region is used as a way-point (`passnode_counts`),
- and the fraction of participants for whom each path is the shortest (`all_path_inclusion_rate`).

## Pipeline
1. **Configure** the target dataset (Language or MD).
2. **Load** healthy connectivity matrices and the AAL parcellation.
3. **For each participant**, compute weighted shortest paths across all AAL node pairs.
4. **Aggregate** paths across participants to produce the corpus DataFrame.


## 1. Imports

Only the libraries actually used downstream:
- `numpy`, `pandas` — array + tabular data handling.
- `scipy.io` — reading the `.mat` connectivity files.
- `bct` — Brain Connectivity Toolbox, for `distance_wei_floyd` and `retrieve_shortest_path`.


In [1]:
import os
import numpy as np
import pandas as pd
import scipy.io as sio
import bct

## 2. Configuration

Set `NETWORK` to either `"Language"` or `"MD"`. All downstream cells read from the
resolved `config` dict, so switching datasets requires only this single edit.

The two datasets differ only in:

| Field | Language | MD |
|---|---|---|
| `column_name` (label in the output DataFrame) | `LanguageNetwork` | `MD` |
| `healthy_data_dir` (where the fMRI-derived matrices live) | `../language_large/…` | `../MD_large/…` |
| `file_tag` (used in output filenames if you save downstream) | `Lang` | `MD` |

Both iterate over the same 116-region AAL parcellation.


In [2]:
# --- Choose the dataset to analyse ---
NETWORK = "Language"   # "Language" or "MD"

NETWORK_CONFIG = {
    "Language": {
        "column_name":      "LanguageNetwork",
        "healthy_data_dir": "language_large/LQT_result/fMRI_Language",
        "file_tag":         "Lang",
    },
    "MD": {
        "column_name":      "MD",
        "healthy_data_dir": "MD_large/LQT_result/fMRI_MD",
        "file_tag":         "MD",
    },
}

config = NETWORK_CONFIG[NETWORK]
COL = config["column_name"]
TAG = config["file_tag"]

# Path to the shared AAL parcellation table
AAL_XLSX = "PhD/sne_measure/aal_table.xlsx"

print(f"Analysing dataset: {NETWORK}  (column='{COL}')")

Analysing dataset: Language  (column='LanguageNetwork')


## 3. Search-information probability

For each shortest path in a connectivity matrix, compute the **search information** —
the number of bits required to specify that path under a random walk on the network,
following Rosvall et al. (2005). Concretely:

1. Build the transition matrix `T = D⁻¹A` (row-normalised connectivity).
2. Find the shortest path `i → … → j` via Floyd–Warshall.
3. Multiply the step-wise transition probabilities along the path to get `P(path | random walk)`.
4. Return `SI[i, j] = −log₂ P(path)` (bits).

Lower `SI` values mean the path is easier to discover by a random walker. If the network
is symmetric, both forward and backward directions are stored (`SI[j, i]`).

`SI` is used in Section 5 to give each shortest path a **discoverability weight** in the corpus.


In [3]:
def search_information_ori(adjacency, transform=None, has_memory=False):
    """Search-information matrix for shortest paths in a weighted network.

    Parameters
    ----------
    adjacency   : (N, N) weighted connectivity matrix.
    transform   : optional weight-to-distance transform passed to `distance_wei_floyd`
                  (e.g. 'inv' for 1/w).
    has_memory  : if True, treat the walker as non-backtracking (Rosvall's memoryful walk).

    Returns
    -------
    SI : (N, N) matrix where SI[i, j] = -log2(prob of the shortest path i -> j).
    """
    N_adj     = len(adjacency)
    flag_triu = bool(np.allclose(adjacency, adjacency.T))

    T = np.linalg.solve(np.diag(np.sum(adjacency, axis=1)), adjacency)
    _, hops, Pmat = bct.distance.distance_wei_floyd(adjacency, transform)

    SI = np.zeros((N_adj, N_adj))
    SI[np.eye(N_adj) > 0] = np.nan

    for i in range(N_adj):
        for j in range(N_adj):
            if (j > i and flag_triu) or (not flag_triu and i != j):
                path = bct.distance.retrieve_shortest_path(i, j, hops, Pmat)
                lp = len(path) - 1
                if flag_triu:
                    if np.any(path):
                        pr_step_ff = np.zeros(lp)
                        pr_step_bk = np.zeros(lp)
                        if has_memory:
                            pr_step_ff[0]    = T[path[0], path[1]]
                            pr_step_bk[lp-1] = T[path[lp], path[lp-1]]
                            for z in range(1, lp):
                                pr_step_ff[z]      = T[path[z], path[z+1]]        / (1 - T[path[z-1], path[z]])
                                pr_step_bk[lp-z-1] = T[path[lp-z], path[lp-z-1]]  / (1 - T[path[lp-z+1], path[lp-z]])
                        else:
                            for z in range(lp):
                                pr_step_ff[z] = T[path[z],   path[z+1]]
                                pr_step_bk[z] = T[path[z+1], path[z]]

                        prob_sp_ff = np.prod(pr_step_ff)
                        prob_sp_bk = np.prod(pr_step_bk)
                        SI[i, j] = -np.log2(prob_sp_ff)
                        SI[j, i] = -np.log2(prob_sp_bk)
                    else:
                        SI[i, j] = np.inf
                        SI[j, i] = np.inf
                else:
                    if np.any(path):
                        pr_step_ff = np.zeros(lp)
                        if has_memory:
                            pr_step_ff[0] = T[path[0], path[1]]
                            for z in range(1, lp):
                                pr_step_ff[z] = T[path[z], path[z+1]] / (1 - T[path[z-1], path[z]])
                        else:
                            for z in range(lp):
                                pr_step_ff[z] = T[path[z], path[z+1]]

                        prob_sp_ff = np.prod(pr_step_ff)
                        SI[i, j] = -np.log2(prob_sp_ff)
                    else:
                        SI[i, j] = np.inf

    return SI

## 4. Data loading

Two inputs are needed to build the corpus:

1. **Healthy connectivity matrices** (one per participant) — the raw material the corpus
   is aggregated from (Section 4.1).
2. **AAL parcellation table** — used to look up human-readable region names for the
   pass-through-node tally (Section 4.2).


### 4.1 Healthy connectivity matrices

For the selected dataset, walk `config["healthy_data_dir"]` and load the per-participant
`{participant}_AAL_percent_parcel_SDC.mat` under each participant's `Parcel_Disconnection/`
folder. Each file provides a 116×116 matrix under the key `pct_sdc_matrix`.

The resulting list `all_data` is iterated over in Section 5.


In [4]:
all_data = []
for folder_name in os.listdir(config["healthy_data_dir"]):
    folder_path = os.path.join(config["healthy_data_dir"], folder_name)
    if not os.path.isdir(folder_path):
        print(f"not a folder: {folder_name}")
        continue
    parcel_folder = os.path.join(folder_path, "Parcel_Disconnection")
    filename      = f"{folder_name}_AAL_percent_parcel_SDC.mat"
    file_path     = os.path.join(parcel_folder, filename)
    mat_data      = sio.loadmat(file_path)
    all_data.append(mat_data['pct_sdc_matrix'])

print(f"Loaded {len(all_data)} healthy connectivity matrices for the {NETWORK} dataset")

not a folder: .DS_Store
Loaded 806 healthy connectivity matrices for the Language dataset


### 4.2 AAL parcellation

- `aal_df` — the 116-region AAL region table (index → region name); used in Section 5
  to record which regions are used as intermediate way-points.
- `aal_nodes` — all AAL indices `[0, 1, …, 115]`. Section 5 iterates over every ordered
  pair `(source, target)` of these to build the full whole-brain shortest-path corpus.


In [5]:
aal_df    = pd.read_excel(AAL_XLSX)
aal_nodes = list(range(len(aal_df)))

print(f"Iterating over {len(aal_nodes)} AAL nodes -> {len(aal_nodes) * (len(aal_nodes) - 1)} ordered source-target pairs")

Iterating over 116 AAL nodes -> 13340 ordered source-target pairs


## 5. Build the path corpus

For every healthy participant and every ordered pair `(source, target)` of AAL nodes:

1. Compute weighted shortest paths in that participant's connectivity matrix using
   Floyd–Warshall (`bct.distance.distance_wei_floyd`, with `transform='inv'` so weight-to-distance
   is `1/w`).
2. Retrieve the shortest path `source → … → target` as a tuple of AAL indices.
3. Update aggregates:
   - `path_counts[path]`     — number of participants whose shortest path from source to target
                                is exactly `path`.
   - `path_prob[path]`       — running sum of search-information probability
                                (normalised to a mean in Section 5.1).
   - `passnode_counts[name]` — count of how often each intermediate region is used as a way-point.

Together these form the **path corpus**: a distribution over shortest paths in a population
of healthy participants, weighted by how discoverable each path is under a random walk.

> **Note.** Rank-deficient matrices are given a tiny diagonal ridge (`1e-7 · I`) so that
> `search_information_ori` (which solves for the transition matrix) stays well-defined.


In [6]:
path_counts     = {}
passnode_counts = {}

for participant_idx, conn_matrix in enumerate(all_data):
    # Weighted shortest paths for this participant
    curSPL, curhops, curPmat = bct.distance.distance_wei_floyd(conn_matrix, 'inv')

    # Add a tiny ridge if the matrix is singular so search_information_ori is well-defined
    if np.linalg.det(conn_matrix) == 0:
        sub_adj = conn_matrix + np.eye(conn_matrix.shape[0]) * 1e-7
    else:
        sub_adj = conn_matrix

    # Iterate every ordered pair of AAL nodes
    for source in aal_nodes:
        for target in aal_nodes:
            if source == target:
                continue
            curpath    = bct.distance.retrieve_shortest_path(source, target, curhops, curPmat)
            path_tuple = tuple(a[0] for a in curpath)

            # Tally intermediate (non-endpoint) regions used as way-points
            for p in curpath:
                if p[0] != source and p[0] != target:
                    region = aal_df['RegionName'][p[0]]
                    passnode_counts[region] = passnode_counts.get(region, 0) + 1

            # Accumulate path counts + search-information probability
            if path_tuple in path_counts:
                path_counts[path_tuple] += 1
            else:
                path_counts[path_tuple] = 1

    # Per-participant progress (much quieter than printing every path_tuple)
    print(f"  participant {participant_idx + 1}/{len(all_data)}: {len(path_counts)} unique paths so far")

  participant 1/806: 10712 unique paths so far
  participant 2/806: 21075 unique paths so far
  participant 3/806: 29581 unique paths so far
  participant 4/806: 35393 unique paths so far
  participant 5/806: 39266 unique paths so far
  participant 6/806: 44314 unique paths so far
  participant 7/806: 48884 unique paths so far
  participant 8/806: 51952 unique paths so far
  participant 9/806: 55804 unique paths so far
  participant 10/806: 59511 unique paths so far
  participant 11/806: 60980 unique paths so far
  participant 12/806: 65458 unique paths so far
  participant 13/806: 68732 unique paths so far
  participant 14/806: 70551 unique paths so far
  participant 15/806: 73134 unique paths so far
  participant 16/806: 75974 unique paths so far
  participant 17/806: 77889 unique paths so far
  participant 18/806: 79350 unique paths so far
  participant 19/806: 82330 unique paths so far
  participant 20/806: 84577 unique paths so far
  participant 21/806: 85653 unique paths so far
 

  participant 168/806: 267113 unique paths so far
  participant 169/806: 267774 unique paths so far
  participant 170/806: 268205 unique paths so far
  participant 171/806: 268275 unique paths so far
  participant 172/806: 269833 unique paths so far
  participant 173/806: 271404 unique paths so far
  participant 174/806: 273201 unique paths so far
  participant 175/806: 274455 unique paths so far
  participant 176/806: 274738 unique paths so far
  participant 177/806: 274905 unique paths so far
  participant 178/806: 275015 unique paths so far
  participant 179/806: 276485 unique paths so far
  participant 180/806: 276592 unique paths so far
  participant 181/806: 277587 unique paths so far
  participant 182/806: 277850 unique paths so far
  participant 183/806: 279003 unique paths so far
  participant 184/806: 279323 unique paths so far
  participant 185/806: 280021 unique paths so far
  participant 186/806: 280849 unique paths so far
  participant 187/806: 281470 unique paths so far


  participant 332/806: 376573 unique paths so far
  participant 333/806: 376927 unique paths so far
  participant 334/806: 377236 unique paths so far
  participant 335/806: 377452 unique paths so far
  participant 336/806: 378017 unique paths so far
  participant 337/806: 378659 unique paths so far
  participant 338/806: 378761 unique paths so far
  participant 339/806: 379034 unique paths so far
  participant 340/806: 379745 unique paths so far
  participant 341/806: 380582 unique paths so far
  participant 342/806: 381179 unique paths so far
  participant 343/806: 382047 unique paths so far
  participant 344/806: 382967 unique paths so far
  participant 345/806: 383061 unique paths so far
  participant 346/806: 383103 unique paths so far
  participant 347/806: 383417 unique paths so far
  participant 348/806: 383515 unique paths so far
  participant 349/806: 384571 unique paths so far
  participant 350/806: 384890 unique paths so far
  participant 351/806: 384902 unique paths so far


  participant 496/806: 454531 unique paths so far
  participant 497/806: 455272 unique paths so far
  participant 498/806: 455545 unique paths so far
  participant 499/806: 455725 unique paths so far
  participant 500/806: 455847 unique paths so far
  participant 501/806: 456473 unique paths so far
  participant 502/806: 456792 unique paths so far
  participant 503/806: 457357 unique paths so far
  participant 504/806: 458015 unique paths so far
  participant 505/806: 459236 unique paths so far
  participant 506/806: 459276 unique paths so far
  participant 507/806: 459759 unique paths so far
  participant 508/806: 459955 unique paths so far
  participant 509/806: 460247 unique paths so far
  participant 510/806: 461054 unique paths so far
  participant 511/806: 461188 unique paths so far
  participant 512/806: 462656 unique paths so far
  participant 513/806: 462760 unique paths so far
  participant 514/806: 462897 unique paths so far
  participant 515/806: 463315 unique paths so far


  participant 660/806: 526801 unique paths so far
  participant 661/806: 526925 unique paths so far
  participant 662/806: 527846 unique paths so far
  participant 663/806: 528445 unique paths so far
  participant 664/806: 528562 unique paths so far
  participant 665/806: 529360 unique paths so far
  participant 666/806: 531549 unique paths so far
  participant 667/806: 531601 unique paths so far
  participant 668/806: 531681 unique paths so far
  participant 669/806: 531915 unique paths so far
  participant 670/806: 532066 unique paths so far
  participant 671/806: 532166 unique paths so far
  participant 672/806: 532380 unique paths so far
  participant 673/806: 532730 unique paths so far
  participant 674/806: 534124 unique paths so far
  participant 675/806: 534211 unique paths so far
  participant 676/806: 534635 unique paths so far
  participant 677/806: 534739 unique paths so far
  participant 678/806: 535243 unique paths so far
  participant 679/806: 535282 unique paths so far


### Path inclusion rate → DataFrame

The **inclusion rate** of a path is the fraction of healthy participants for whom that
path is the (weighted) shortest path between its endpoints:

$$
\text{inclusion\_rate}(\text{path}) = \frac{\text{path\_counts}[\text{path}]}{n_{\text{participants}}}
$$

Store the result as a DataFrame keyed by the path tuple, with the dataset name (`COL`)
as the column header. This form makes the two datasets trivially joinable side-by-side
(index-align two DataFrames on the path tuple).


In [8]:
all_path_inclusion_rate = {
    path_tuple: path_counts[path_tuple] / len(all_data)
    for path_tuple in path_counts
}

df_all_inclusionRate = pd.DataFrame.from_dict(
    all_path_inclusion_rate, orient='index', columns=[COL])

print(f"Corpus size: {len(df_all_inclusionRate)} unique paths")


Corpus size: 582006 unique paths
